# nexgddp_hyras_bc_zarr — Debug Notebook
Step-by-step version of `nexgddp_hyras_bc_zarr.py` for interactive inspection.
Each section corresponds to a logical block in the script.

## 1. Imports

In [1]:
import os
import shutil
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr

from climdata.sdba.bcsd import BiasCorrection
from climdata._vendor.isimip3basd import utility_functions as uf
from climdata._vendor.isimip3basd import bias_adjustment as ba

print('imports OK')

imports OK


## 2. Configuration
Edit these instead of command-line args.

In [2]:
MODEL      = 'ACCESS-CM2'
MEMBER     = 'r1i1p1f1'
VARIABLES  = ['pr', 'tas', 'tasmax', 'tasmin', 'rsds', 'hurs']
SCENARIO   = 'ssp370'   # change per run; historical / ssp126 / ssp370

N_ITERATIONS = 20
N_PROCESSES  = 20        # keep 1 for debugging; increase for production
SEED         = 42

ZARR_DIR   = '/data01/FDS/muduchuru/Atmos/zarr_stores'
OUTPUT_DIR = '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV'

REF_START  = '1951-01-01'
REF_END    = '1980-12-31'
HIST_START = '1951-01-01'
HIST_END   = '2014-12-31'
FUT_START  = '2015-01-01'
FUT_END    = '2100-12-31'

VAR_RENAME = {
    'pr':     'Precipitation',
    'tas':    'TempMean',
    'tasmax': 'TempMax',
    'tasmin': 'TempMin',
    'rsds':   'Radiation',
    'hurs':   'RelHumCalc',
}

OUTPUT_COLUMNS = [
    'Precipitation',
    'TempMin',
    'TempMean',
    'TempMax',
    'Radiation',
    'SunshineDuration',
    'SoilMoisture',
    'SoilTemperature',
    'Windspeed',
    'RefETcalc',
    'RefETdwd',
    'RelHumCalc',
]

## 3. Helper functions

In [3]:
def to_standard_units(ds: xr.Dataset) -> xr.Dataset:
    """K → degC for temperature; kg m-2 s-1 → mm d-1 for pr."""
    ds = ds.copy()
    for var in ('tas', 'tasmax', 'tasmin'):
        if var not in ds:
            continue
        u = str(ds[var].attrs.get('units', '')).strip().upper()
        if u in ('K', 'KELVIN'):
            ds[var] = (ds[var] - 273.15).assign_attrs({**ds[var].attrs, 'units': 'degC'})
            print(f'  [{var}] K → degC')
    if 'pr' in ds:
        u = str(ds['pr'].attrs.get('units', '')).strip().lower()
        if 'kg' in u:
            ds['pr'] = (ds['pr'] * 86400.0).assign_attrs({**ds['pr'].attrs, 'units': 'mm d-1'})
            print('  [pr] kg m-2 s-1 → mm d-1')
    return ds


def rename_and_convert(row_data: dict) -> dict:
    """Rename CF names and convert units for CSV output.
    rsds: W m-2 → kJ m-2 (multiply by 86.4 = 86400 s/day / 1000)
    """
    out = {}
    for var, arr in row_data.items():
        if var == 'rsds':
            arr = arr * 86.4
        out[VAR_RENAME.get(var, var)] = arr
    return out


def obs_zarr_path(zarr_dir):
    return os.path.join(zarr_dir, 'HYRAS_REC2D_obs_1951-01-01_2014-12-31.zarr')


def sim_zarr_path(zarr_dir, model, scenario):
    if scenario == 'historical':
        return os.path.join(zarr_dir, f'NEX_GDDP_{model}_historical_1951_2014.zarr')
    return os.path.join(zarr_dir, f'NEX_GDDP_{model}_{scenario}_2015_2100.zarr')


def open_zarr(store_path, start_date, end_date, variables):
    if not os.path.exists(store_path):
        raise FileNotFoundError(f'Zarr store not found: {store_path}')
    print(f'  {os.path.basename(store_path)}')
    ds = xr.open_zarr(store_path, consolidated=True, chunks='auto')
    keep    = [v for v in variables if v in ds.data_vars]
    missing = set(variables) - set(keep)
    if missing:
        print(f'  WARNING: variables missing in store: {sorted(missing)}')
    ds = ds[keep].sel(time=slice(start_date, end_date))
    print(f'  dims : {dict(ds.dims)}')
    print(f'  vars : {list(ds.data_vars)}')
    return ds

print('helpers defined')

helpers defined


## 4. Open HYRAS obs Zarr

In [4]:
obs_raw = open_zarr(obs_zarr_path(ZARR_DIR), HIST_START, HIST_END, VARIABLES)
obs_ref = obs_raw.sel(time=slice(REF_START, REF_END))
print(f'obs_ref dims : {dict(obs_ref.dims)}')
obs_ref

  HYRAS_REC2D_obs_1951-01-01_2014-12-31.zarr
  dims : {'time': 23376, 'lat': 794, 'lon': 1006}
  vars : ['pr', 'tas', 'tasmax', 'tasmin', 'rsds', 'hurs']
obs_ref dims : {'time': 10958, 'lat': 794, 'lon': 1006}


<xarray.Dataset> Size: 210GB
Dimensions:  (time: 10958, lat: 794, lon: 1006)
Coordinates:
  * lat      (lat) float64 6kB 47.16 47.17 47.18 47.19 ... 55.07 55.08 55.09
  * lon      (lon) float64 8kB 5.46 5.47 5.48 5.49 ... 15.48 15.49 15.5 15.51
  * time     (time) datetime64[ns] 88kB 1951-01-01 1951-01-02 ... 1980-12-31
Data variables:
    pr       (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tas      (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tasmax   (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    rsds     (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    hurs     (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
Attributes: (12/24)
    CDI:                    Climate Data Interface version 2.5.0 (https://mpi...
    CDO:                    Climate Data Operators version 2.5.0 (https://mpi...
    Conventions:            CF-1.11
    ConventionsURL:         http://cfconventions.org/Data/cf-conventions/cf-c...
    author:                 Hydrometeorology (KU41)
    contact:                hydromet@dwd.de
    ...                     ...
    realm:                  atmos
    references:             https://opendata.dwd.de/climate_environment/CDC/g...
    source:                 surface observations, satellite observations (SIS...
    title:                  gridded_global_shortwave_radiation_dataset_(HYRAS...
    unique_dataset_id:      DWD_HYRAS_DE_5km_rsds_v3-1_1951_day_006718A87E
    variable_id:            rsds

In [5]:
# Spot-check one variable — should already be in degC / mm
for var in VARIABLES:
    if var in obs_ref:
        da = obs_ref[var]
        sample = da.isel(time=0).values
        print(f'  {var:10s}  units={da.attrs.get("units","?")}  '
              f'min={np.nanmin(sample):.2f}  max={np.nanmax(sample):.2f}  '
              f'nans={np.isnan(sample).sum()}')

  pr          units=mm  min=0.00  max=34.11  nans=338846
  tas         units=degree_Celsius  min=-11.49  max=5.21  nans=338846


KeyboardInterrupt: 

## 5. Open sim historical Zarr

In [6]:
sim_hist_full = open_zarr(
    sim_zarr_path(ZARR_DIR, MODEL, 'historical'), HIST_START, HIST_END, VARIABLES)
sim_ref = to_standard_units(sim_hist_full.sel(time=slice(REF_START, REF_END)))
print(f'sim_ref dims : {dict(sim_ref.dims)}')
sim_ref

  NEX_GDDP_ACCESS-CM2_historical_1951_2014.zarr
  dims : {'time': 23376, 'lat': 794, 'lon': 1006}
  vars : ['pr', 'tas', 'tasmax', 'tasmin', 'rsds', 'hurs']
  [tas] K → degC
  [tasmax] K → degC
  [tasmin] K → degC
  [pr] kg m-2 s-1 → mm d-1
sim_ref dims : {'time': 10958, 'lat': 794, 'lon': 1006}


<xarray.Dataset> Size: 210GB
Dimensions:  (time: 10958, lat: 794, lon: 1006)
Coordinates:
  * lat      (lat) float64 6kB 47.16 47.17 47.18 47.19 ... 55.07 55.08 55.09
  * lon      (lon) float64 8kB 5.46 5.47 5.48 5.49 ... 15.48 15.49 15.5 15.51
  * time     (time) datetime64[ns] 88kB 1951-01-01 1951-01-02 ... 1980-12-31
Data variables:
    pr       (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tas      (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tasmax   (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    rsds     (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
    hurs     (time, lat, lon) float32 35GB dask.array<chunksize=(10958, 10, 10), meta=np.ndarray>
Attributes: (12/24)
    CDI:                   Climate Data Interface version 2.5.0 (https://mpim...
    CDO:                   Climate Data Operators version 2.5.0 (https://mpim...
    Conventions:           CF-1.7
    activity:              NEX-GDDP-CMIP6
    cmip6_institution_id:  CSIRO-ARCCSS
    cmip6_license:         CC-BY-SA 4.0
    ...                    ...
    scenario:              historical
    source:                BCSD
    title:                 ACCESS-CM2, r1i1p1f1, historical, global downscale...
    tracking_id:           51b6f31d-fba7-4c62-ae19-92575c59630d
    variant_label:         r1i1p1f1
    version:               2.0

In [ ]:
# Spot-check after unit conversion
for var in VARIABLES:
    if var in sim_ref:
        da = sim_ref[var]
        sample = da.isel(time=0).values
        print(f'  {var:10s}  units={da.attrs.get("units","?")}  '
              f'min={np.nanmin(sample):.2f}  max={np.nanmax(sample):.2f}  '
              f'nans={np.isnan(sample).sum()}')

  pr          units=mm d-1  min=0.00  max=14.64  nans=81382
  tas         units=degC  min=-11.80  max=4.28  nans=81382
  tasmax      units=degC  min=-8.31  max=7.28  nans=81382
  tasmin      units=degC  min=-15.30  max=2.38  nans=81382
  rsds        units=W m-2  min=12.52  max=87.93  nans=81382
  hurs        units=%  min=79.55  max=95.80  nans=81382


## 6. Open sim future Zarr (or use historical)

In [7]:
if SCENARIO == 'historical':
    sim_fut = to_standard_units(sim_hist_full)
else:
    sim_fut = to_standard_units(
        open_zarr(
            sim_zarr_path(ZARR_DIR, MODEL, SCENARIO),
            FUT_START, FUT_END, VARIABLES,
        )
    )
print(f'sim_fut dims : {dict(sim_fut.dims)}')
sim_fut

  NEX_GDDP_ACCESS-CM2_ssp370_2015_2100.zarr
  dims : {'time': 31411, 'lat': 794, 'lon': 1006}
  vars : ['pr', 'tas', 'tasmax', 'tasmin', 'rsds', 'hurs']
  [tas] K → degC
  [tasmax] K → degC
  [tasmin] K → degC
  [pr] kg m-2 s-1 → mm d-1
sim_fut dims : {'time': 31411, 'lat': 794, 'lon': 1006}


<xarray.Dataset> Size: 602GB
Dimensions:  (time: 31411, lat: 794, lon: 1006)
Coordinates:
  * lat      (lat) float64 6kB 47.16 47.17 47.18 47.19 ... 55.07 55.08 55.09
  * lon      (lon) float64 8kB 5.46 5.47 5.48 5.49 ... 15.48 15.49 15.5 15.51
  * time     (time) datetime64[ns] 251kB 2015-01-01 2015-01-02 ... 2100-12-31
Data variables:
    pr       (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
    tas      (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
    tasmax   (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
    tasmin   (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
    rsds     (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
    hurs     (time, lat, lon) float32 100GB dask.array<chunksize=(31411, 10, 10), meta=np.ndarray>
Attributes: (12/25)
    CDI:                   Climate Data Interface version 2.5.0 (https://mpim...
    CDO:                   Climate Data Operators version 2.5.0 (https://mpim...
    Conventions:           CF-1.7
    activity:              NEX-GDDP-CMIP6
    cmip6_institution_id:  CSIRO-ARCCSS
    cmip6_license:         CC-BY-SA 4.0
    ...                    ...
    scenario:              ssp370
    source:                BCSD
    title:                 ACCESS-CM2, r1i1p1f1, ssp370, global downscaled CM...
    tracking_id:           a43ca3bf-6569-4b56-8d7d-79bf9cefd7da
    variant_label:         r1i1p1f1
    version:               2.0

## 7. BiasCorrection initialisation

In [8]:
bc = BiasCorrection(
    variable           = VARIABLES,
    n_iterations       = N_ITERATIONS,
    n_processes        = N_PROCESSES,
    randomization_seed = SEED,
)

BiasCorrection initialized for ['pr', 'tas', 'tasmax', 'tasmin', 'rsds', 'hurs']
  n_iterations=20 (MBCn multivariate)  n_processes=20  seed=42
  [pr] dist=gamma  trend=mixed  detrend=False  adjust_p=True
  [tas] dist=normal  trend=additive  detrend=True  adjust_p=False
  [tasmax] dist=normal  trend=additive  detrend=True  adjust_p=False
  [tasmin] dist=normal  trend=additive  detrend=True  adjust_p=False
  [rsds] dist=beta  trend=bounded  detrend=False  adjust_p=True
  [hurs] dist=beta  trend=bounded  detrend=False  adjust_p=True


## 8. Convert to iris cubes
Inspect the cubes before they enter the vendor code.

In [9]:
obs_cubes, sh_cubes, sf_cubes = bc._to_iris_cubes(obs_ref, sim_ref, sim_fut)

print(f'obs_cubes : {len(obs_cubes)} cube(s)')
for i, c in enumerate(obs_cubes):
    print(f'  [{i}] shape={c.shape}  dtype={c.dtype}  '
          f'calendar={c.coord("time").units.calendar}')

print(f'sh_cubes  : {len(sh_cubes)} cube(s)')
for i, c in enumerate(sh_cubes):
    print(f'  [{i}] shape={c.shape}')

print(f'sf_cubes  : {len(sf_cubes)} cube(s)')
for i, c in enumerate(sf_cubes):
    print(f'  [{i}] shape={c.shape}')

obs_cubes : 6 cube(s)
  [0] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
  [1] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
  [2] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
  [3] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
  [4] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
  [5] shape=(10958, 794, 1006)  dtype=float32  calendar=proleptic_gregorian
sh_cubes  : 6 cube(s)
  [0] shape=(10958, 794, 1006)
  [1] shape=(10958, 794, 1006)
  [2] shape=(10958, 794, 1006)
  [3] shape=(10958, 794, 1006)
  [4] shape=(10958, 794, 1006)
  [5] shape=(10958, 794, 1006)
sf_cubes  : 6 cube(s)
  [0] shape=(31411, 794, 1006)
  [1] shape=(31411, 794, 1006)
  [2] shape=(31411, 794, 1006)
  [3] shape=(31411, 794, 1006)
  [4] shape=(31411, 794, 1006)
  [5] shape=(31411, 794, 1006)


In [10]:
# Inspect lat/lon DimCoords on first obs cube
c = obs_cubes[0]
print('obs_cubes[0] coords:')
for coord in c.coords():
    print(f'  {coord.name():20s}  shape={coord.shape}  '
          f'units={coord.units}  axis={coord.standard_name}')

obs_cubes[0] coords:
  time                  shape=(10958,)  units=hours since 1931-01-01T00:00:00+00:00  axis=time
  latitude              shape=(794,)  units=degrees_north  axis=latitude
  longitude             shape=(1006,)  units=degrees_east  axis=longitude


## 9. Prepare BA state
Calls `ba.initializer` and builds lazy_data / rotation_matrices.

In [11]:
lazy_data, month_numbers, years, doys, space_shape, rotation_matrices = \
    bc._prepare_ba_state(obs_cubes, sh_cubes, sf_cubes)

print(f'space_shape       : {space_shape}')
print(f'n_locations       : {int(np.prod(space_shape)):,}')
print(f'rotation_matrices : {len(rotation_matrices)} matrices, each {rotation_matrices[0].shape}')
print()
print('lazy_data keys and per-key shapes:')
for key, arrs in lazy_data.items():
    print(f'  {key:12s}  {len(arrs)} array(s)  shapes={[a.shape for a in arrs]}')
print()
print('month_numbers:', {k: (v.shape if v is not None else None) for k, v in month_numbers.items()})
print('years        :', {k: (v.shape if v is not None else None) for k, v in years.items()})
print('doys         :', {k: (v.shape if v is not None else None) for k, v in doys.items()})

space_shape       : (794, 1006)
n_locations       : 798,764
rotation_matrices : 20 matrices, each (6, 6)

lazy_data keys and per-key shapes:
  obs_hist      6 array(s)  shapes=[(10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006)]
  sim_hist      6 array(s)  shapes=[(10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006), (10958, 794, 1006)]
  sim_fut       6 array(s)  shapes=[(31411, 794, 1006), (31411, 794, 1006), (31411, 794, 1006), (31411, 794, 1006), (31411, 794, 1006), (31411, 794, 1006)]

month_numbers: {'obs_hist': (10958,), 'sim_hist': (10958,), 'sim_fut': (31411,)}
years        : {'obs_hist': (10958,), 'sim_hist': (10958,), 'sim_fut': (31411,)}
doys         : {'obs_hist': (10958,), 'sim_hist': (10958,), 'sim_fut': (31411,)}


In [12]:
# Inspect a slice of the lazy obs_hist data to verify values are sensible
sample = lazy_data['obs_hist'][0][(slice(0, 10),) + tuple(0 for _ in space_shape)].compute()
print('obs_hist var-0, loc (0,0), first 10 timesteps:', sample)

obs_hist var-0, loc (0,0), first 10 timesteps: [-- -- -- -- -- -- -- -- -- --]


## 10. npy_stack setup

In [13]:
out_dir = os.path.join(OUTPUT_DIR, MODEL, SCENARIO)
os.makedirs(out_dir, exist_ok=True)

n_times    = month_numbers['sim_fut'].size
time_coord = sf_cubes[0].coord('time')
time_dates = [str(d) for d in time_coord.units.num2date(time_coord.points)]

tmpdir    = Path(out_dir) / '_tmp_npy'
tmpdir.mkdir(parents=True, exist_ok=True)
npy_paths = [(tmpdir / f'ba_{v}.nc').resolve() for v in bc.variables]

for p in npy_paths:
    uf.setup_npy_stack(str(p), (n_times,) + space_shape)

print(f'n_times   : {n_times}')
print(f'npy_paths : {[p.name for p in npy_paths]}')
print(f'npy dirs  : {[uf.npy_stack_dir(str(p)) for p in npy_paths]}')

n_times   : 31411
npy_paths : ['ba_pr.nc', 'ba_tas.nc', 'ba_tasmax.nc', 'ba_tasmin.nc', 'ba_rsds.nc', 'ba_hurs.nc']
npy dirs  : ['/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_pr.nc.npy_stack/', '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_tas.nc.npy_stack/', '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_tasmax.nc.npy_stack/', '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_tasmin.nc.npy_stack/', '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_rsds.nc.npy_stack/', '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy/ba_hurs.nc.npy_stack/']


## 11. Row / col index maps

In [14]:
lat_name = 'lat' if 'lat' in obs_ref.coords else 'latitude'
lon_name = 'lon' if 'lon' in obs_ref.coords else 'longitude'
lats = obs_ref[lat_name].values
lons = obs_ref[lon_name].values

lat_to_row = {float(lat): i for i, lat in enumerate(np.sort(lats)[::-1])}
lon_to_col = {float(lon): j for j, lon in enumerate(np.sort(lons))}

start_str = str(sim_fut.time.values[0])[:10].replace('-', '')
end_str   = str(sim_fut.time.values[-1])[:10].replace('-', '')

print(f'lat range : {lats.min():.2f} → {lats.max():.2f}  ({len(lats)} pts)')
print(f'lon range : {lons.min():.2f} → {lons.max():.2f}  ({len(lons)} pts)')
print(f'time range: {start_str} → {end_str}')

lat range : 47.16 → 55.09  (794 pts)
lon range : 5.46 → 15.51  (1006 pts)
time range: 20150101 → 21001231


## 12. Single-location dry-run
Run bias correction for one cell and inspect the raw npy result before writing CSV.
Useful for diagnosing BA output issues without iterating all locations.

In [15]:
# Pick a location — change to any valid (row, col) within space_shape
DEBUG_LOC = (400, 400)  # (lat-index, lon-index)

print(f'Running correct_one_location for i_loc={DEBUG_LOC} ...')
t0 = time.time()
bc.correct_one_location(DEBUG_LOC, space_shape, npy_paths, rotation_matrices)
print(f'Done in {time.time()-t0:.1f}s')

Running correct_one_location for i_loc=(400, 400) ...
(400, 400)
Done in 3.5s


In [17]:
i_1d = np.ravel_multi_index(DEBUG_LOC, space_shape)
col_data = {}
for var, p in zip(bc.variables, npy_paths):
    npy_file = uf.npy_stack_dir(str(p)) + f'{i_1d}.npy'
    arr = np.load(npy_file).squeeze()
    col_data[var] = arr
    print(f'  {var:10s}  shape={arr.shape}  '
          f'min={np.nanmin(arr):.4f}  max={np.nanmax(arr):.4f}  '
          f'nans={np.isnan(arr).sum()}')
col_data = rename_and_convert(col_data)

  pr          shape=(31411,)  min=0.0000  max=73.2326  nans=0
  tas         shape=(31411,)  min=-14.0092  max=33.0241  nans=0
  tasmax      shape=(31411,)  min=-11.3118  max=43.0085  nans=0
  tasmin      shape=(31411,)  min=-20.5009  max=25.8878  nans=0
  rsds        shape=(31411,)  min=1.3229  max=350.8093  nans=0
  hurs        shape=(31411,)  min=25.6254  max=98.7444  nans=0


In [18]:
df_debug = (
    pd.DataFrame({'time': time_dates, **col_data})
    .reindex(columns=['time'] + OUTPUT_COLUMNS)
)
print(df_debug.head(10))
df_debug.describe()

                  time  Precipitation   TempMin  TempMean   TempMax  \
0  2015-01-01 00:00:00       4.100646  0.132957  4.373941  7.825685   
1  2015-01-02 00:00:00       0.000000 -2.277729  0.271230  4.868227   
2  2015-01-03 00:00:00       0.000000 -7.775842 -3.721689 -2.487269   
3  2015-01-04 00:00:00       0.882258 -8.319579 -4.613566 -0.307805   
4  2015-01-05 00:00:00       4.451818 -0.608630  0.446937  3.276752   
5  2015-01-06 00:00:00       0.000000 -6.237339 -4.200546 -2.941404   
6  2015-01-07 00:00:00       0.000000 -7.652256 -5.185585 -3.370629   
7  2015-01-08 00:00:00       2.299763 -8.493152 -6.152712 -1.538605   
8  2015-01-09 00:00:00       0.145013 -3.066267 -0.770437  0.856750   
9  2015-01-10 00:00:00       3.703369 -2.154890  0.435099  1.256092   

     Radiation  SunshineDuration  SoilMoisture  SoilTemperature  Windspeed  \
0  1978.014404               NaN           NaN              NaN        NaN   
1  1756.056030               NaN           NaN              Na

,Precipitation,TempMin,TempMean,TempMax,Radiation,SunshineDuration,SoilMoisture,SoilTemperature,Windspeed,RefETcalc,RefETdwd,RelHumCalc
count,31411.000000,31411.000000,31411.000000,31411.000000,31411.000000,0.0,0.0,0.0,0.0,0.0,0.0,31411.000000
mean,1.788170,7.047544,11.315260,15.820147,10596.752930,NaN,NaN,NaN,NaN,NaN,NaN,75.467972
std,3.963856,6.900577,7.793062,9.266600,7546.089355,NaN,NaN,NaN,NaN,NaN,NaN,11.553731
min,0.000000,-20.500948,-14.009181,-11.311774,114.302086,NaN,NaN,NaN,NaN,NaN,NaN,25.625446
25%,0.000000,1.891549,5.509616,8.757676,3642.613892,NaN,NaN,NaN,NaN,NaN,NaN,67.141354
50%,0.000000,7.189562,11.417653,16.010731,9218.514648,NaN,NaN,NaN,NaN,NaN,NaN,77.174988
75%,1.840222,12.390113,17.417283,23.013129,16355.874023,NaN,NaN,NaN,NaN,NaN,NaN,84.848511
max,73.232574,25.887838,33.024075,43.008541,30309.923828,NaN,NaN,NaN,NaN,NaN,NaN,98.744408


In [19]:
# Quick time-series plot (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(len(bc.variables), 1, figsize=(14, 2.5 * len(bc.variables)),
                             sharex=True)
    if len(bc.variables) == 1:
        axes = [axes]
    out_vars = [VAR_RENAME.get(v, v) for v in bc.variables]
    for ax, var in zip(axes, out_vars):
        ax.plot(df_debug['time'].values[::10], df_debug[var].values[::10],
                linewidth=0.5)
        ax.set_ylabel(var)
        ax.set_title(f'{var}  loc={DEBUG_LOC}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available — skip plot')

KeyboardInterrupt: 

Error in callback <function flush_figures at 0x1479b165bb50> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

## 13. Write debug CSV

In [21]:
lat_dbg = float(lats[DEBUG_LOC[0]])
lon_dbg = float(lons[DEBUG_LOC[1]])
row_num = lat_to_row[lat_dbg]
col_num = lon_to_col[lon_dbg]

col_dir  = Path(out_dir) / str(col_num)
col_dir.mkdir(parents=True, exist_ok=True)
csv_path = col_dir / f'{MODEL}_{SCENARIO}_{start_str}_{end_str}_C{col_num}R{row_num}.csv'

df_debug.to_csv(csv_path, index=False)
print(f'Written: {csv_path}')

Written: /data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/400/ACCESS-CM2_ssp370_20150101_21001231_C400R393.csv


## 14. Full serial loop (small subset)
Run all locations serially. For a full run use the script; this cell is for testing a
small sub-region. Set `MAX_LOCS` to limit the number of cells processed.

In [ ]:
MAX_LOCS = 10000  # set to None to run all locations

n_total = int(np.prod(space_shape))
n_done = n_skipped = 0
t0 = time.time()

for idx, i_loc in enumerate(np.ndindex(space_shape)):
    if MAX_LOCS is not None and idx >= MAX_LOCS:
        print(f'Stopped after {MAX_LOCS} locations (MAX_LOCS limit)')
        break

    lat = float(lats[i_loc[0]])
    lon = float(lons[i_loc[1]])
    row_num = lat_to_row[lat]
    col_num = lon_to_col[lon]

    col_dir  = Path(out_dir) / str(col_num)
    col_dir.mkdir(parents=True, exist_ok=True)
    csv_path = col_dir / f'{MODEL}_{SCENARIO}_{start_str}_{end_str}_C{col_num}R{row_num}.csv'

    if csv_path.exists():
        n_skipped += 1
        continue

    bc.correct_one_location(i_loc, space_shape, npy_paths, rotation_matrices)

    i_1d = np.ravel_multi_index(i_loc, space_shape)
    npy_file_0 = uf.npy_stack_dir(str(npy_paths[0])) + f'{i_1d}.npy'
    if not os.path.exists(npy_file_0):
        # vendor skipped this cell (all-missing data) — no output to write
        continue

    row_data = {}
    for var, p in zip(bc.variables, npy_paths):
        npy_file = uf.npy_stack_dir(str(p)) + f'{i_1d}.npy'
        row_data[var] = np.load(npy_file).squeeze()
        os.remove(npy_file)

    (
        pd.DataFrame({'time': time_dates, **rename_and_convert(row_data)})
        .reindex(columns=['time'] + OUTPUT_COLUMNS)
        .to_csv(csv_path, index=False)
    )
    n_done += 1

    elapsed = time.time() - t0
    rate    = n_done / elapsed if elapsed > 0 else 0
    print(f'  {idx+1:>5}/{n_total}  i_loc={i_loc}  C{col_num}R{row_num}  '
          f'{rate:.2f} loc/s  {csv_path.name}')

print(f'\nDone: {n_done} written, {n_skipped} skipped  ({time.time()-t0:.1f}s total)')

Stopped after 10000 locations (MAX_LOCS limit)

Done: 0 written, 10000 skipped  (1.0s total)


In [24]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import numpy as np
import pandas as pd

def process_location(args):
    """Worker function to process a single location using fast thread-shared memory."""
    idx, i_loc, space_shape, npy_paths, rotation_matrices, lats, lons, lat_to_row, lon_to_col, out_dir, time_dates, output_cols = args

    lat = float(lats[i_loc[0]])
    lon = float(lons[i_loc[1]])
    row_num = lat_to_row[lat]
    col_num = lon_to_col[lon]

    # Directory is guaranteed to exist now; no locks needed
    csv_path = Path(out_dir) / str(col_num) / f'{MODEL}_{SCENARIO}_{start_str}_{end_str}_C{col_num}R{row_num}.csv'

    if csv_path.exists():
        return "skipped", idx, i_loc, col_num, row_num, csv_path.name

    # Core mathematical operation (GIL released during heavy numpy operations)
    bc.correct_one_location(i_loc, space_shape, npy_paths, rotation_matrices)

    i_1d = np.ravel_multi_index(i_loc, space_shape)
    npy_file_0 = uf.npy_stack_dir(str(npy_paths[0])) + f'{i_1d}.npy'
    if not os.path.exists(npy_file_0):
        return "vendor_skipped", idx, i_loc, col_num, row_num, csv_path.name

    row_data = {}
    for var, p in zip(bc.variables, npy_paths):
        npy_file = uf.npy_stack_dir(str(p)) + f'{i_1d}.npy'
        try:
            row_data[var] = np.load(npy_file).squeeze()
            os.remove(npy_file)
        except FileNotFoundError:
            pass

    # Build payload, reindex columns, and handle dummy NaNs natively
    df_payload = rename_and_convert(row_data)
    df = pd.DataFrame({'time': time_dates, **df_payload}).reindex(columns=['time'] + output_cols)
    
    # Fast vectorized fill for non-calculated dummy columns
    dummy_cols = [c for c in output_cols if c not in df_payload]
    if dummy_cols:
        df[dummy_cols] = -999.0

    df.to_csv(csv_path, index=False)
    return "done", idx, i_loc, col_num, row_num, csv_path.name


# --- Main Execution Block ---

MAX_LOCS = 10000   # Set to None to run all locations
NUM_THREADS = 40   # Matches your hardware scaling limit

n_total = int(np.prod(space_shape))
n_done = n_skipped = n_vendor_skipped = 0
t0 = time.time()

# 1. Pre-create directories instantly on the main thread to eliminate thread locks
print("Pre-building output directory tree...")
unique_cols = set(lon_to_col.values())
for col_num in unique_cols:
    (Path(out_dir) / str(col_num)).mkdir(parents=True, exist_ok=True)

# 2. Package tasks efficiently
tasks = []
for idx, i_loc in enumerate(np.ndindex(space_shape)):
    if MAX_LOCS is not None and idx >= MAX_LOCS:
        print(f'Stopped generating tasks at {MAX_LOCS} locations (MAX_LOCS limit)')
        break

    tasks.append((
        idx, i_loc, space_shape, npy_paths, rotation_matrices,
        lats, lons, lat_to_row, lon_to_col, out_dir, time_dates, OUTPUT_COLUMNS
    ))

print(f"Launching ThreadPoolExecutor ({NUM_THREADS} threads) for {len(tasks)} locations...")

# 3. Process with a clean, throttled reporting loop
t_last_report = time.time()

with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    # Use submit + as_completed to keep track of progress smoothly without terminal lag
    future_to_task = {executor.submit(process_location, task): task for task in tasks}
    
    for i, future in enumerate(as_completed(future_to_task)):
        status, idx, i_loc, col_num, row_num, csv_name = future.result()

        if status == "skipped":
            n_skipped += 1
        elif status == "vendor_skipped":
            n_vendor_skipped += 1
        elif status == "done":
            n_done += 1

        # Throttle printing to avoid filling up standard output buffers (updates every 500 tasks or 5 seconds)
        now = time.time()
        if i % 500 == 0 or (now - t_last_report) > 5 or (i + 1) == len(tasks):
            elapsed = now - t0
            rate = (n_done + n_skipped + n_vendor_skipped) / elapsed if elapsed > 0 else 0
            remaining = len(tasks) - (i + 1)
            eta_min = (remaining / rate / 60) if rate > 0 else 0
            
            print(f" Progress: {i+1:>6}/{len(tasks)} | Done: {n_done:,} | Skipped: {n_skipped:,} | "
                  f"Speed: {rate:.1f} loc/s | ETA: {eta_min:.1f} min")
            t_last_report = now

print(f'\n Run Complete: {n_done} written, {n_skipped} skipped, {n_vendor_skipped} vendor-empty.')
print(f' Total Time: {time.time() - t0:.1f} seconds.')

Pre-building output directory tree...
Stopped generating tasks at 10000 locations (MAX_LOCS limit)
Launching ThreadPoolExecutor (40 threads) for 10000 locations...
 Progress:      1/10000 | Done: 0 | Skipped: 1 | Speed: 0.4 loc/s | ETA: 473.8 min
(0, 207) skipped due to missing data
(0, 215) skipped due to missing data
(0, 229) skipped due to missing data
(0, 222) skipped due to missing data
(0, 227) skipped due to missing data
(0, 214) skipped due to missing data
(0, 237) skipped due to missing data
(0, 204) skipped due to missing data
(0, 213) skipped due to missing data
(0, 223) skipped due to missing data
(0, 203) skipped due to missing data
(0, 238) skipped due to missing data
(0, 231) skipped due to missing data
(0, 219) skipped due to missing data
(0, 221) skipped due to missing data
(0, 202) skipped due to missing data
(0, 218) skipped due to missing data
(0, 239) skipped due to missing data
(0, 216) skipped due to missing data
(0, 205) skipped due to missing data
(0, 220) skip

OSError: Cannot save file into a non-existent directory: '/data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/226'

## 15. Cleanup tmp npy files

In [ ]:
# Run this cell to remove the temporary npy_stack directory
shutil.rmtree(str(tmpdir), ignore_errors=True)
print(f'Removed: {tmpdir}')

Removed: /data01/FDS/muduchuru/Atmos/NEXGDDP_HYRAS_BC_CSV/ACCESS-CM2/ssp370/_tmp_npy
